# SoundStream — analysis

Дополнение к Comet Report: [soundstream-final](https://www.comet.com/dmitrii-trusov/soundstream-neural-audio-codec/wlqcmxsufeozaosf8crhkaod3ailgtyp)

## Обучение и эксперименты

### Данные
- Train: LibriSpeech `train-clean-100`
- Test: LibriSpeech `test-clean`
- Sample rate: 16 kHz, mono

### Конфиг обучения (`soundstream.yaml`)
- Crop на train: 0.5 s, batch 12, Adam lr 2e-4
- Эпохи: 100 × 450 шагов
- Encoder strides `[2, 4, 5, 5]`, RVQ 8 × codebook 1024
- Loss G: reconstruction + adversarial + feature matching (λ=100) + commitment

### Runs (Comet)
- `soundstream-dev` — отладка пайплайна
- `soundstream-final` — финальное обучение, чекпоинт `checkpoint-epoch100.pth`

### Железо
- Финальный train: **NVIDIA RTX 5090**, ~45k шагов (100 эпох × 450) — ушло относительно быстро

### Замечания по обучению
- На старте ловил несовпадения длин после decoder (crop train vs полный utterance на eval) — поправил обрезку/pad под длину входа
- `loss_commitment` на test выше, чем на train-crop — разная длина сегмента, смотреть train и test отдельно
- RVQ: инициализация кодбука k-means + EMA; без сильного feature matching (λ=100) GAN давал нестабильный звук
- Отдельные loss и perplexity в Comet — удобно ловить расхождение G/D

### Метрики на всём `test-clean` (`evaluate.py`)
- STOI: **0.8245**
- NISQA MOS: **2.443**

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torchaudio
from IPython.display import Audio, display
from hydra import compose, initialize_config_dir

root = Path.cwd().parent if not (Path.cwd() / "src").is_dir() else Path.cwd()
sys.path.insert(0, str(root))

from evaluate import get_device, load_generator, reconstruct_batch
from inference import load_audio
from src.datasets.speech_dataset import SpeechDataset

with initialize_config_dir(version_base=None, config_dir=str(root / "src/configs")):
    cfg = compose(config_name="evaluate")
device = get_device()
fs = cfg.audio.sample_rate
gen = load_generator(cfg, device)
test_ds = SpeechDataset(str(root / cfg.data.test_root), fs, is_train=False, crop_seconds=cfg.audio.crop_seconds)
mel = torchaudio.transforms.MelSpectrogram(fs, n_fft=1024, hop_length=256, n_mels=80)

def show(orig, title):
    audio = orig if orig.dim() == 3 else orig.unsqueeze(0)
    recon = reconstruct_batch(gen, audio, device)
    o, r = audio.squeeze(0).cpu(), recon.squeeze(0).cpu()
    fig, ax = plt.subplots(2, 2, figsize=(9, 5))
    ax[0, 0].plot(o.squeeze()); ax[0, 1].plot(r.squeeze())
    ax[1, 0].imshow(mel(o).log1p().squeeze(), aspect="auto", origin="lower")
    ax[1, 1].imshow(mel(r).log1p().squeeze(), aspect="auto", origin="lower")
    fig.suptitle(title); plt.tight_layout(); plt.show()
    display(Audio(o.squeeze(), rate=fs)); display(Audio(r.squeeze(), rate=fs))

## Qualitative analysis, in-domain

### Протокол
- Датасет: `test-clean`
- 3 utterance (начало / середина / конец индексации)
- Сравнение: waveform, mel, audio (original / reconstructed)

### Ячейка ниже
- Реконструкция через `checkpoint-epoch100.pth`

In [ ]:
for i in [0, len(test_ds) // 2, len(test_ds) - 1]:
    show(test_ds[i]["audio"], Path(test_ds[i]["audio_path"]).name)

## External Dataset Analysis — English

### Протокол
- Датасет: LJ Speech (не LibriSpeech)
- URL: `LJ025-0076.wav`
- Train модели: только LibriSpeech

### Ячейка ниже
- Тот же пайплайн: waveform, mel, audio

In [ ]:
show(load_audio("https://keithito.com/LJ-Speech-Dataset/LJ025-0076.wav", fs), "LJ Speech")

## External Dataset Analysis — Russian

### Протокол
- Пример: [russian-single-speaker-speech-dataset](https://huggingface.co/datasets/niobures/russian-single-speaker-speech-dataset) (чтение на русском, HF)
- Train модели: LibriSpeech (английский)

### Ячейка ниже
- Тот же пайплайн: waveform, mel, audio

In [ ]:
RU_URL = "https://huggingface.co/datasets/niobures/russian-single-speaker-speech-dataset/resolve/main/early_short_stories/early_short_stories_0001.wav"
show(load_audio(RU_URL, fs), "Russian")

## Quantitative analysis

Метрики на **всем** `test-clean` (`evaluate.py` → Comet):

| Метрика | Значение |
|---------|----------|
| STOI | 0.8245 |
| NISQA MOS | 2.443 |

## Выводы

Латент ~6 kbps против 256 kbps PCM — память по битам сжалась примерно в 40 раз; по слуху и mel речь в целом нормально восстанавливается (STOI 0.82), но слышен фоновый шум и лёгкие артефакты RVQ, на LJ и русском чуть хуже из‑за другого домена; шумы, скорее всего, уйдут, если увеличить ёмкость кодека — больше квантайзеров и codebook.
